# 02c. De novo transcriptome assembly with Trinity and Salmon

このノートブックは、定量化パターン3「referenceなしでtranscriptomeを組み立て、その組み立て結果にreadをmappingして定量する」方法である。

**今どの解析段階にいるか**

FASTQ readからtranscript候補を組み立て、そのde novo transcriptomeを参照としてSalmonで定量する段階である。

**このノートブックの役割**

- 参照genome/transcriptomeを使わずに、全サンプルのreadからtranscript候補を組み立てる。
- Trinity由来のtranscriptomeからSalmon indexを作り、各サンプルを定量する。
- transcript単位とTrinity gene cluster単位のcount matrixを作る。

**入力**

- `metadata/samples.tsv`（各サンプルの `sample_id`, 群情報, R1/R2 FASTQパスを持つ表）
- paired-end FASTQファイル（`samples.tsv` の `fastq_1`, `fastq_2` が指すreadファイル）

**出力**

- `results/denovo/trinity/Trinity.fasta`（readから組み立てたtranscript候補配列）
- `results/denovo/trinity_tx2gene.tsv`（Trinity transcript IDとTrinity gene cluster IDの対応表）
- `results/quant/denovo_salmon/<sample_id>/quant.sf`（de novo transcriptomeに対するSalmon定量結果）
- `results/counts/transcript_counts_denovo_trinity.tsv`（行=Trinity transcript、列=sampleのcount matrix）
- `results/counts/gene_counts_denovo_trinity.tsv`（行=Trinity gene cluster、列=sampleのcount matrix）

**次に進む条件**

- Trinity assemblyの `Trinity.fasta` がある。
- 9サンプルすべてにde novo Salmonの `quant.sf` がある。
- count matrixの列名が `samples.tsv` の `sample_id` と一致している。

**重要な注意**

このデータはヒトBCBL-1細胞由来と考えられるため、通常はGENCODEなどの参照を使うパターン2またはパターン1の方が解釈しやすい。de novo assemblyは、よい参照がない生物や未知transcript探索の学習用ルートとして扱う。


### `salmon quant` コマンドライン引数の説明

| 引数 | 意味 |
|------|------|
| `-i DIR` | `salmon index` で作成したindex directoryを指定する。ここではTrinity由来のde novo transcriptome indexである。 |
| `-l A` | Library type（ライブラリ形式）の指定。`A` はAutomatic（自動判定）であり、strandnessが不明な場合に扱いやすい。 |
| `-1 FILE` | ペアエンドのR1（mate 1）FASTQファイルパスである。 |
| `-2 FILE` | ペアエンドのR2（mate 2）FASTQファイルパスである。 |
| `-p N` | 並列スレッド数。定量に使うCPU数である。 |
| `--validateMappings` | mappingの検証を行うフラグ。alignment-based定量に近い精度が得られる。 |
| `-o DIR` | 出力ディレクトリ（`results/quant/denovo_salmon/<sample_id>/`）を指定する。 |

Salmonは**quasi-mapping**（擬似mapping）によりFASTQ readをTranscriptに対応づけ、各Transcriptの発現量を**TPM（Transcripts Per Million）**および**NumReads**（推定read数）として出力する。
主な出力ファイル `quant.sf` の列は次の通りである。

| 列名 | 意味 |
|------|------|
| `Name` | Transcript ID |
| `Length` | Transcript全長（bp）|
| `EffectiveLength` | 読まれ得る有効長 |
| `TPM` | 長さ・sequencing深度補正後の発現量 |
| `NumReads` | Transcriptへの推定read数（DESeq2に渡す値）|


### Trinity コマンドライン引数の説明

| 引数 | 意味 |
|------|------|
| `--seqType fq` | 入力ファイル形式。`fq` はFASTQ形式である。 |
| `--left FILES` | ペアエンドのR1（mate 1）FASTQファイルリストをカンマ区切りで指定する。 |
| `--right FILES` | ペアエンドのR2（mate 2）FASTQファイルリストをカンマ区切りで指定する。 |
| `--CPU N` | 並列スレッド数。 |
| `--max_memory N` | Trinityが使用する最大メモリ量（例: `10G`）を指定する。 |
| `--output DIR` | アセンブリ結果の出力ディレクトリを指定する。 |
| `--SS_lib_type RF` | 任意のstrand-specific指定。このノートブックの現在のコマンドには入れていないため、ライブラリの鎖特異性が確実に分かっている場合だけ追加する。 |

Trinityは**de novo transcriptome assembly**を行い、reference genomeなしにreadだけからtranscript候補配列を再構築する。
構築されたtranscriptomeに対してSalmonで発現定量を行うことで、reference-freeなRNA-seq解析が可能である。


In [ ]:
from pathlib import Path
import re
import shutil
import subprocess
import time

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_DIR = Path('/Users/yusuke_tateishi/Documents/RNA_seq').resolve()
SAMPLES = pd.read_csv(PROJECT_DIR / 'metadata/samples.tsv', sep='\t')

TRINITY_OUT_DIR = PROJECT_DIR / 'results/denovo/trinity'
TRINITY_FASTA = PROJECT_DIR / 'results/denovo/trinity/Trinity.fasta'
TRINITY_TX2GENE_PATH = PROJECT_DIR / 'results/denovo/trinity_tx2gene.tsv'
DENOVO_SALMON_INDEX = PROJECT_DIR / 'results/denovo/salmon_index'
DENOVO_QUANT_DIR = PROJECT_DIR / 'results/quant/denovo_salmon'
OUT_TRANSCRIPT_MATRIX = PROJECT_DIR / 'results/counts/transcript_counts_denovo_trinity.tsv'
OUT_GENE_MATRIX = PROJECT_DIR / 'results/counts/gene_counts_denovo_trinity.tsv'

THREADS = 8
TRINITY_MAX_MEMORY = '50G'
LIBRARY_TYPE = 'A'

SAMPLES[['sample_id', 'condition', 'fastq_1', 'fastq_2']]


## 解析ルートの全体像

パターン3では、最初にreadだけからtranscript候補を組み立てる。その後、その組み立て結果を「自分で作ったtranscriptome reference」としてSalmonで定量する。

```text
FASTQ
  -> Trinity de novo assembly
  -> Trinity.fasta
  -> Salmon index
  -> Salmon quantification
  -> transcript count matrix
  -> Trinity gene cluster count matrix
```

de novo assemblyの基本は、readを短いk-merに分解し、k-mer同士の重なりからtranscript配列を復元することである。単純化すると、1つのreadは次のように分解される。

$$
\mathrm{read} = (k_1, k_2, \ldots, k_m)
$$

ここで $k_i$ はread内の短い部分配列である。Trinityは、多数のreadから得たk-merのつながりを使ってtranscript候補を作る。


## ツールが見えるか確認する

`Trinity` はde novo assembly、`salmon` は組み立てたtranscriptomeへの定量に使う。

この環境ではTrinityが未導入の可能性がある。未導入でも、まずコマンドの意味と出力の形を確認できる。


In [ ]:
tool_names = ['Trinity', 'salmon']
tool_status = pd.DataFrame(
    [{'tool': name, 'path': shutil.which(name), 'available': shutil.which(name) is not None} for name in tool_names]
)
tool_status


## 入力FASTQと出力予定を確認する

de novo assemblyでは、参照FASTAやGTFは使わない。代わりに、全sampleのFASTQ readを材料にして `Trinity.fasta` を作る。

出力されるTrinity IDは、たとえば次のような形である。

```text
TRINITY_DN12345_c0_g1_i1
TRINITY_DN12345_c0_g1_i2
```

`i1` と `i2` はisoform候補、`g1` までがTrinity内のgene cluster候補である。これは `TP53` のようなヒト遺伝子名ではない。


In [ ]:
fastq_status = []
for _, sample in SAMPLES.iterrows():
    fastq_1 = PROJECT_DIR / sample['fastq_1']
    fastq_2 = PROJECT_DIR / sample['fastq_2']
    fastq_status.append(
        {
            'sample_id': sample['sample_id'],
            'fastq_1_exists': fastq_1.exists(),
            'fastq_2_exists': fastq_2.exists(),
            'fastq_1': str(fastq_1.relative_to(PROJECT_DIR)),
            'fastq_2': str(fastq_2.relative_to(PROJECT_DIR)),
        }
    )

output_status = pd.DataFrame(
    [
        {'name': 'TRINITY_FASTA', 'path': TRINITY_FASTA.relative_to(PROJECT_DIR), 'exists': TRINITY_FASTA.exists()},
        {'name': 'TRINITY_TX2GENE_PATH', 'path': TRINITY_TX2GENE_PATH.relative_to(PROJECT_DIR), 'exists': TRINITY_TX2GENE_PATH.exists()},
        {'name': 'DENOVO_SALMON_INDEX', 'path': DENOVO_SALMON_INDEX.relative_to(PROJECT_DIR), 'exists': DENOVO_SALMON_INDEX.exists()},
    ]
)

display(pd.DataFrame(fastq_status))
display(output_status)


## Trinity assemblyコマンドを作る

Trinityには、全sampleのR1 FASTQを `--left`、全sampleのR2 FASTQを `--right` として渡す。

ここで作る `Trinity.fasta` は、以後のSalmon定量で使うtranscriptome referenceである。パターン2のGENCODE transcript FASTAとは違い、このプロジェクトのreadから推定した配列になる。


In [ ]:
left_fastqs = [str(PROJECT_DIR / path) for path in SAMPLES['fastq_1']]
right_fastqs = [str(PROJECT_DIR / path) for path in SAMPLES['fastq_2']]

trinity_command = [
    'Trinity',
    '--seqType', 'fq',
    '--left', ','.join(left_fastqs),
    '--right', ','.join(right_fastqs),
    '--CPU', str(THREADS),
    '--max_memory', TRINITY_MAX_MEMORY,
    '--output', str(TRINITY_OUT_DIR),
]

print(' '.join(trinity_command[:10]), '...')
print('left FASTQ files:', len(left_fastqs))
print('right FASTQ files:', len(right_fastqs))
print('Expected assembly:', TRINITY_FASTA.relative_to(PROJECT_DIR))


## Trinity assemblyを実行する

初回だけ `RUN_TRINITY = True` にする。de novo assemblyは非常に重く、メモリと時間を使う。

このセルは、Trinityが導入され、FASTQがすべて揃っていることを確認してから実行しよう。


In [ ]:
RUN_TRINITY = False

if RUN_TRINITY:
    trinity = shutil.which('Trinity')
    if trinity is None:
        raise RuntimeError('Trinity was not found. Install Trinity before running de novo assembly.')
    missing_fastqs = [path for path in left_fastqs + right_fastqs if not Path(path).exists()]
    if missing_fastqs:
        raise FileNotFoundError(f'Missing FASTQ files: {missing_fastqs[:3]}')

    TRINITY_OUT_DIR.parent.mkdir(parents=True, exist_ok=True)
    command = [trinity] + trinity_command[1:]
    print('Running:', ' '.join(command[:10]), '...')
    start_time = time.time()
    subprocess.run(command, check=True)
    print(f'Trinity run time: {time.time() - start_time:.2f} seconds')
else:
    print('Set RUN_TRINITY = True only after Trinity and all FASTQ files are ready.')


## Trinity transcript IDからtx2geneを作る

Salmonはtranscript単位で `NumReads` を出する。Trinityでは複数isoform候補を同じgene clusterにまとめたいので、次の対応表を作る。

```text
transcript_id              gene_id
TRINITY_DN12345_c0_g1_i1   TRINITY_DN12345_c0_g1
TRINITY_DN12345_c0_g1_i2   TRINITY_DN12345_c0_g1
```

これはGENCODEの `tx2gene.tsv` と同じ役割だが、`gene_id` はヒト遺伝子名ではなくTrinity gene cluster IDである。


In [ ]:
def extract_trinity_gene_id(transcript_id: str) -> str:
    match = re.match(r'^(TRINITY_DN\d+_c\d+_g\d+)_i\d+', transcript_id)
    if match:
        return match.group(1)
    return transcript_id.rsplit('_i', 1)[0] if '_i' in transcript_id else transcript_id

WRITE_TRINITY_TX2GENE = False

if TRINITY_FASTA.exists():
    transcript_ids = []
    with TRINITY_FASTA.open() as handle:
        for line in handle:
            if line.startswith('>'):
                transcript_ids.append(line[1:].split()[0])

    tx2gene = pd.DataFrame(
        {
            'transcript_id': transcript_ids,
            'gene_id': [extract_trinity_gene_id(transcript_id) for transcript_id in transcript_ids],
        }
    )
    display(tx2gene.head())
    print('transcripts:', len(tx2gene))
    print('gene clusters:', tx2gene['gene_id'].nunique())

    if WRITE_TRINITY_TX2GENE:
        TRINITY_TX2GENE_PATH.parent.mkdir(parents=True, exist_ok=True)
        tx2gene.to_csv(TRINITY_TX2GENE_PATH, sep='\t', index=False)
        print('Wrote:', TRINITY_TX2GENE_PATH.relative_to(PROJECT_DIR))
    else:
        print('Set WRITE_TRINITY_TX2GENE = True after reviewing the parsed IDs.')
else:
    example_tx2gene = pd.DataFrame(
        {
            'transcript_id': ['TRINITY_DN12345_c0_g1_i1', 'TRINITY_DN12345_c0_g1_i2'],
            'gene_id': ['TRINITY_DN12345_c0_g1', 'TRINITY_DN12345_c0_g1'],
        }
    )
    display(example_tx2gene)
    print('No Trinity.fasta yet:', TRINITY_FASTA.relative_to(PROJECT_DIR))


## Salmon index作成コマンドを作る

ここではGENCODE transcript FASTAではなく、Trinityが作った `Trinity.fasta` からSalmon indexを作る。

つまり、パターン2の「既知transcript辞書」ではなく、パターン3の「自分のreadから作ったtranscript辞書」を使う。


### `salmon index` コマンドライン引数の説明

| 引数 | 意味 |
|------|------|
| `-t FILE` | index構築に使うtranscript FASTAを指定する。ここではTrinityが作った `Trinity.fasta` である。 |
| `-i DIR` | 作成したSalmon indexの出力先ディレクトリを指定する。 |
| `-p N` | index構築に使う並列スレッド数。 |

GENCODE routeでは既知transcript配列からindexを作るが、このreference-free routeでは自分のreadから推定したtranscript候補をindex化する。


In [ ]:
salmon_index_command = [
    'salmon',
    'index',
    '-t', str(TRINITY_FASTA),
    '-i', str(DENOVO_SALMON_INDEX),
    '-p', str(THREADS),
]

print(' '.join(salmon_index_command))


## de novo Salmon indexを作成する

初回だけ `RUN_SALMON_INDEX = True` にする。

`Trinity.fasta` がまだない場合は、このセルは実行できない。先にTrinity assemblyを完了する。


In [ ]:
RUN_SALMON_INDEX = False

if RUN_SALMON_INDEX:
    salmon = shutil.which('salmon')
    if salmon is None:
        raise RuntimeError('salmon was not found. Activate the rna-seq environment first.')
    if not TRINITY_FASTA.exists():
        raise FileNotFoundError(TRINITY_FASTA)

    DENOVO_SALMON_INDEX.parent.mkdir(parents=True, exist_ok=True)
    command = [salmon] + salmon_index_command[1:]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Set RUN_SALMON_INDEX = True only after Trinity.fasta exists.')


## Salmon quantificationコマンドを作る

各sampleのFASTQを、Trinity由来のtranscriptome indexへmappingして定量する。

Salmon出力 `quant.sf` の代表的な列は次の通りである。

```text
Name                         Length  EffectiveLength  TPM   NumReads
TRINITY_DN12345_c0_g1_i1     900     720              12.3  154.7
```

count matrix作成には、主に `Name` と `NumReads` を使う。


In [ ]:
salmon_quant_commands = []
for _, sample in SAMPLES.iterrows():
    output_dir = DENOVO_QUANT_DIR / sample['sample_id']
    command = [
        'salmon',
        'quant',
        '-i', str(DENOVO_SALMON_INDEX),
        '-l', LIBRARY_TYPE,
        '-1', str(PROJECT_DIR / sample['fastq_1']),
        '-2', str(PROJECT_DIR / sample['fastq_2']),
        '-p', str(THREADS),
        '--validateMappings',
        '-o', str(output_dir),
    ]
    salmon_quant_commands.append(command)

print('First de novo Salmon command:')
print(' '.join(salmon_quant_commands[0]))


## de novo Salmon quantificationを実行する

初回だけ `RUN_SALMON_QUANT = True` にする。

9 sampleすべてに `quant.sf` ができたら、transcript count matrixとTrinity gene cluster count matrixを作れる。


In [ ]:
RUN_SALMON_QUANT = False

if RUN_SALMON_QUANT:
    salmon = shutil.which('salmon')
    if salmon is None:
        raise RuntimeError('salmon was not found. Activate the rna-seq environment first.')
    if not DENOVO_SALMON_INDEX.exists():
        raise FileNotFoundError(DENOVO_SALMON_INDEX)

    start_time = time.time()
    for command in salmon_quant_commands:
        output_dir = Path(command[-1])
        output_dir.mkdir(parents=True, exist_ok=True)
        run_command = [salmon] + command[1:]
        print('Running:', ' '.join(run_command))
        subprocess.run(run_command, check=True)
    print(f'Total de novo Salmon run time: {time.time() - start_time:.2f} seconds')
else:
    print('Set RUN_SALMON_QUANT = True only after the de novo Salmon index is ready.')


## Salmon結果がそろっているか確認する

各sampleの `quant.sf` が存在すれば、count matrixを作れる。


In [ ]:
quant_status = []
for sample_id in SAMPLES['sample_id']:
    quant_path = DENOVO_QUANT_DIR / sample_id / 'quant.sf'
    quant_status.append(
        {
            'sample_id': sample_id,
            'quant_sf_exists': quant_path.exists(),
            'path': str(quant_path.relative_to(PROJECT_DIR)),
        }
    )

quant_status = pd.DataFrame(quant_status)
display(quant_status)

existing_quant_paths = [DENOVO_QUANT_DIR / sample_id / 'quant.sf' for sample_id in SAMPLES['sample_id'] if (DENOVO_QUANT_DIR / sample_id / 'quant.sf').exists()]
if existing_quant_paths:
    example_quant = pd.read_csv(existing_quant_paths[0], sep='\t', usecols=['Name', 'Length', 'EffectiveLength', 'TPM', 'NumReads'])
    display(example_quant.sort_values(by='NumReads', ascending=False).head())
else:
    print('No de novo quant.sf files yet.')


## transcript count matrixとgene cluster count matrixを作る

まずtranscript単位では、各 `quant.sf` の `NumReads` を横に並べる。

```text
transcript_id                  Non_1  Non_2  Oxi_2h_1
TRINITY_DN12345_c0_g1_i1       100    120    80
TRINITY_DN12345_c0_g1_i2       20     15     10
```

次に、同じTrinity gene clusterに属するtranscriptを足し合わせる。

$$
\mathrm{count}_{G,s} = \sum_{t \in G} \mathrm{NumReads}_{t,s}
$$

- `G` はTrinity gene clusterである。例: `TRINITY_DN12345_c0_g1`
- `t` はそのclusterに属するtranscriptである。例: `..._i1`, `..._i2`
- `s` はsampleである。
- `NumReads_t,s` は各sampleの `quant.sf` にある `NumReads` である。

小さい例では、`g1_i1 = 100`、`g1_i2 = 20` なら、

$$
\mathrm{count}_{g1,Non\_1} = 100 + 20 = 120
$$

になる。


In [ ]:
BUILD_COUNT_MATRIX = False

if BUILD_COUNT_MATRIX:
    if TRINITY_TX2GENE_PATH.exists():
        tx2gene = pd.read_csv(TRINITY_TX2GENE_PATH, sep='\t')
    elif TRINITY_FASTA.exists():
        transcript_ids = []
        with TRINITY_FASTA.open() as handle:
            for line in handle:
                if line.startswith('>'):
                    transcript_ids.append(line[1:].split()[0])
        tx2gene = pd.DataFrame(
            {
                'transcript_id': transcript_ids,
                'gene_id': [extract_trinity_gene_id(transcript_id) for transcript_id in transcript_ids],
            }
        )
    else:
        raise FileNotFoundError(TRINITY_TX2GENE_PATH)

    required_columns = {'transcript_id', 'gene_id'}
    if not required_columns.issubset(tx2gene.columns):
        raise ValueError(f'tx2gene must contain columns: {required_columns}')
    tx2gene = tx2gene[['transcript_id', 'gene_id']].drop_duplicates()

    per_sample_transcript_counts = []
    per_sample_gene_counts = []
    for sample_id in SAMPLES['sample_id']:
        quant_path = DENOVO_QUANT_DIR / sample_id / 'quant.sf'
        if not quant_path.exists():
            raise FileNotFoundError(quant_path)

        quant = pd.read_csv(quant_path, sep='\t', usecols=['Name', 'NumReads']).rename(columns={'Name': 'transcript_id'})
        transcript_counts = quant.set_index('transcript_id')['NumReads']
        transcript_counts.name = sample_id
        per_sample_transcript_counts.append(transcript_counts)

        merged = quant.merge(tx2gene, on='transcript_id', how='inner')
        gene_counts = merged.groupby('gene_id', as_index=True)['NumReads'].sum()
        gene_counts.name = sample_id
        per_sample_gene_counts.append(gene_counts)

    transcript_matrix = pd.concat(per_sample_transcript_counts, axis=1).fillna(0).round().astype(int)
    transcript_matrix.insert(0, 'transcript_id', transcript_matrix.index)
    OUT_TRANSCRIPT_MATRIX.parent.mkdir(parents=True, exist_ok=True)
    transcript_matrix.to_csv(OUT_TRANSCRIPT_MATRIX, sep='\t', index=False)

    gene_matrix = pd.concat(per_sample_gene_counts, axis=1).fillna(0).round().astype(int)
    gene_matrix.insert(0, 'gene_id', gene_matrix.index)
    OUT_GENE_MATRIX.parent.mkdir(parents=True, exist_ok=True)
    gene_matrix.to_csv(OUT_GENE_MATRIX, sep='\t', index=False)

    print('Wrote:', OUT_TRANSCRIPT_MATRIX.relative_to(PROJECT_DIR))
    print('Wrote:', OUT_GENE_MATRIX.relative_to(PROJECT_DIR))
    display(gene_matrix.head())
else:
    print('Set BUILD_COUNT_MATRIX = True after all de novo quant.sf files exist.')


## count matrixの形を確認する

count matrixができたら、列名が `samples.tsv` の `sample_id` と一致しているか確認する。

注意: `gene_counts_denovo_trinity.tsv` の `gene_id` は `TRINITY_...` IDである。clusterProfilerでGO解析をするには、別途BLASTやTransDecoderなどでヒトgene IDへannotationする必要がある。


In [ ]:
for matrix_name, matrix_path, id_column in [
    ('transcript matrix', OUT_TRANSCRIPT_MATRIX, 'transcript_id'),
    ('gene cluster matrix', OUT_GENE_MATRIX, 'gene_id'),
]:
    print('---', matrix_name, '---')
    if matrix_path.exists():
        counts = pd.read_csv(matrix_path, sep='\t')
        print('shape:', counts.shape)
        print('columns:', list(counts.columns[:10]))
        display(counts.head())

        missing_samples = set(SAMPLES['sample_id']) - set(counts.columns)
        extra_samples = set(counts.columns) - {id_column} - set(SAMPLES['sample_id'])
        print('missing_samples:', sorted(missing_samples))
        print('extra_samples:', sorted(extra_samples))
    else:
        print('No matrix yet:', matrix_path.relative_to(PROJECT_DIR))


**よくある間違い**

- ヒトのように参照がよく整備された生物で、理由なくde novo assemblyを主解析にしてしまう。
- Trinity IDをそのまま `TP53` や `JUN` のような既知gene名として解釈してしまう。
- `Trinity.fasta` を作るsample群と、Salmonで定量するsample群を混ぜてしまう。
- assemblyが重いため、メモリ不足や途中停止に気づかず次へ進んでしまう。

**小さい練習**

`example_tx2gene` の2行を見て、`TRINITY_DN12345_c0_g1_i1` と `TRINITY_DN12345_c0_g1_i2` が同じgene clusterに足し合わされる理由を説明しよう。
